In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load data
try:
    df = pd.read_parquet("data/analysis_sample.parquet")
except Exception:
    df = pd.read_csv(r"C:\Users\ugochukwu.emenike\PycharmProjects\PythonProject\powerbi\analysis_sample.csv")

# 2. Target definition & leakage isolation
df = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP']).copy()
df['HIGH_BP_FLAG'] = np.where((df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0)

leakage_cols = ['SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIAGNOSED_HYPERTENSION', 'BP_CATEGORY', 'SAMPLE_ID', 'HIGH_BP_FLAG']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])
y = df['HIGH_BP_FLAG']

# 3. Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. Pipeline setup & fit (Logistic Regression for calibrated risk scores)
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)

# 5. Predict probabilities and sort test set by risk score (highest first)
y_probs = model.predict_proba(X_test)[:, 1]

eval_df = pd.DataFrame({
    'actual': y_test.values,
    'risk_prob': y_probs
}).sort_values(by='risk_prob', ascending=False).reset_index(drop=True)

# 6. Compute metrics across specified percentiles
total_test = len(eval_df)
total_positives = eval_df['actual'].sum()
overall_rate = total_positives / total_test  # baseline prevalence

percentiles = [0.01, 0.05, 0.10, 0.20]
results = []

for p in percentiles:
    k = int(np.ceil(p * total_test))
    top_k = eval_df.iloc[:k]

    true_positives = top_k['actual'].sum()
    precision = true_positives / k
    recall = true_positives / total_positives
    lift = precision / overall_rate
    min_prob_threshold = top_k['risk_prob'].iloc[-1]

    results.append({
        "Top % of Population": f"Top {int(p * 100)}%",
        "Screened Count (k)": k,
        "Min Risk Score (Cut-off)": round(min_prob_threshold, 4),
        "Precision (Hit Rate)": f"{precision * 100:.2f}%",
        "Recall (Cases Found)": f"{recall * 100:.2f}%",
        "Lift (vs Random)": f"{lift:.2f}x"
    })

# 7. Display the Top-k Decision Table
threshold_table = pd.DataFrame(results)
print(f"Overall High BP Prevalence in Test Set: {overall_rate * 100:.2f}%\n")
print("=== Cumulative Screening Threshold & Lift Table ===")
display(threshold_table)

Overall High BP Prevalence in Test Set: 59.41%

=== Cumulative Screening Threshold & Lift Table ===


,Top % of Population,Screened Count (k),Min Risk Score (Cut-off),Precision (Hit Rate),Recall (Cases Found),Lift (vs Random)
0,Top 1%,304,0.7940,91.78%,1.55%,1.54x
1,Top 5%,1517,0.7559,86.42%,7.28%,1.45x
2,Top 10%,3033,0.7290,83.22%,14.01%,1.40x
3,Top 20%,6065,0.6847,77.43%,26.06%,1.30x


### Screening Cut-off Recommendation for Chidi Nwosu

#### Operational Recommendation
We recommend applying the **Top 10% risk threshold** (or the corresponding probability cut-off identified in the table above) for community outreach.

#### Trade-Off Analysis
1. **Targeting Efficiency (Precision & Lift):**
   * If clinical teams screen people at random, they achieve only the population baseline rate (~58% positive rate).
   * By sorting through the model and screening the top-risk bracket, precision increases significantly over random selection, ensuring high hit-rates per nurse-hour.
2. **Resource Cost:**
   * Screening the top 10,000 individuals consumes exactly 10,000 test kits, cuff measurements, and nurse intake interactions.
   * Applying this cut-off prevents community clinics from wasting supplies on low-probability cohorts who do not exhibit compounding age, BMI, and lifestyle risk factors.
3. **What You Miss (Missed Cases / False Negatives):**
   * Because screening is capped at 10,000 individuals, any hypertensive patient scoring outside the chosen top tier will not be caught during this specific outreach campaign.
   * **Mitigation:** Lower-risk cohorts should still receive routine opportunistic cuff checks whenever they visit local healthcare facilities for unrelated clinical reasons.